# Serving a GenAI app over REST — models-from-code

*Part of the `gen_ai/` track. Prerequisites: `c_genai_evaluation`, `e_prompt_registry`. The traditional-ML analog is `ml/g_model_serving`.*

## The problem: an app only your Python process can call isn't deployed

In `a_`–`e_` the app ran **in this notebook**. To let *other* services use it — a web backend, a batch job, a teammate — it needs to sit behind a **REST endpoint**. That's the same step `ml/g_model_serving` took for an sklearn model: **log it as an MLflow model, then `mlflow models serve` it.**

But a GenAI app can't be pickled — it's *code* plus live clients, not a fitted array. MLflow's answer is **models-from-code**: you log a small Python **script** that rebuilds the app, and MLflow serves that. The app loads `qa-answer@production` (from `e_`) **at request time**, so promoting a new prompt version reaches the live endpoint with no redeploy.

| Term | One-line meaning |
|---|---|
| **models-from-code** | Log a `.py` that builds the model (via `mlflow.models.set_model(...)`) instead of pickling an object. |
| **pyfunc** | MLflow's universal model interface — `predict(model_input)` — the same one `ml/` models use. |
| **`/invocations`** | The REST endpoint `mlflow models serve` exposes; POST JSON in, `{"predictions": [...]}` out. |

## Prerequisites

- **Tracking server on 5001**, **Ollama** (`qwen3:8b`), and the **`qa-answer` prompt registered** with a `@production` alias — run `e_prompt_registry` first.
- We'll start a **model server on port 5002** (a second process, like `ml/g_model_serving`).
- No new dependencies.

## Step 1 — write the app as a model script

`%%writefile` drops the app into `_genai_app.py`. It's an ordinary `pyfunc` model: `load_context` builds the Ollama client once; `predict` loads `qa-answer@production`, fills in each question, and calls the model. The final `mlflow.models.set_model(...)` is what makes it loggable **from code**.

In [1]:
%%writefile _genai_app.py
import os, re
import mlflow
from openai import OpenAI

class QAApp(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        self._client = OpenAI(
            base_url=os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
            api_key="ollama", timeout=60, max_retries=0)
        self._model = os.environ.get("OLLAMA_MODEL", "qwen3:8b")

    def predict(self, context, model_input, params=None):
        # model_input arrives as a list of strings (or a pandas Series/DataFrame over REST)
        if hasattr(model_input, "iloc"):
            col = model_input.iloc[:, 0] if getattr(model_input, "ndim", 1) > 1 else model_input
            questions = col.tolist()
        else:
            questions = list(model_input)
        # loaded per request -> moving the @production alias changes behaviour with no redeploy
        prompt = mlflow.genai.load_prompt("prompts:/qa-answer@production")
        out = []
        for q in questions:
            r = self._client.chat.completions.create(
                model=self._model, max_tokens=512,
                messages=[{"role": "user", "content": prompt.format(question=str(q)) + " /no_think"}])
            out.append(re.sub(r"<think>.*?</think>", "", r.choices[0].message.content, flags=re.DOTALL).strip())
        return out

mlflow.models.set_model(QAApp())

Writing _genai_app.py


## Step 2 — log it as an MLflow model

`python_model="_genai_app.py"` is the models-from-code part: MLflow stores the script (and an `input_example` to infer the signature), not a pickle. Copy the printed **model URI** — you'll serve it next.

In [2]:
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("genai-serving")

with mlflow.start_run():
    info = mlflow.pyfunc.log_model(
        name="qa_app",
        python_model="_genai_app.py",
        input_example=["What is MLflow Tracing?"],
        pip_requirements=["mlflow", "openai"],
    )
print("model URI:", info.model_uri)

# Copy-paste the next line into a SEPARATE terminal to serve it:
print("\nServe with:\n")
print(f'MLFLOW_TRACKING_URI=http://127.0.0.1:5001 \\\n'
      f'  mlflow models serve -m "{info.model_uri}" '
      f'-p 5002 --host 127.0.0.1 --env-manager local')

/home/targ/venv_projects/teach-mlflow/.venv/lib/python3.14/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/06/05 16:45:08 INFO mlflow.pyfunc: Inferring model signature from input example
2026/06/05 16:45:28 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
/home/targ/venv_projects/teach-mlflow/.venv/lib/python3.14/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.

🏃 View run unequaled-gnat-248 at: http://127.0.0.1:5001/#/experiments/17/runs/d3abd1d6b4a94c239bcde35dd8451dfb
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/17
model URI: models:/m-da091854429944b5b641f4455722cbbb


## Step 3 — sanity-check in-process

Before standing up a server, load the logged model as a pyfunc and predict — the same `models:/...` artifact the server will use. (Cross-link `ml/g_model_serving`, which did the identical in-process-vs-REST check for sklearn.)

In [3]:
loaded = mlflow.pyfunc.load_model(info.model_uri)
print(loaded.predict(["What does mlflow models serve do?"]))

/home/targ/venv_projects/teach-mlflow/.venv/lib/python3.14/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


['MLflow Models Serve is a tool for deploying machine learning models into production, enabling them to be accessed via a REST API for making predictions.']


## Step 4 — serve it over REST

A served model is a **separate process**. Two ways to start it:

**A — Automated, from this notebook (no copy-paste).** The next cell launches `mlflow models serve` as a background subprocess using the `info.model_uri` **variable**, waits for `/health`, and you stop it in the teardown cell at the end. Handy for running the tutorial.

**B — A separate terminal (how you'd deploy for real).** Paste the command the **log cell printed** (URI already filled in):

```bash
MLFLOW_TRACKING_URI=http://127.0.0.1:5001 \
  mlflow models serve -m "models:/<URI>" -p 5002 --host 127.0.0.1 --env-manager local
```

Either way, two flags matter:

- **`MLFLOW_TRACKING_URI=...`** — the served app calls `load_prompt(...)` at request time, so the server must reach the registry.
- **`--env-manager local`** — serve in *this* environment (fast); drop it and MLflow builds a fresh virtualenv from the model's logged requirements (reproducible but slow).

In [ ]:
# Option A: launch the server from the notebook using the info.model_uri variable.
import os, subprocess, time, requests

SERVE_URL = "http://127.0.0.1:5002"
server = subprocess.Popen(
    ["mlflow", "models", "serve", "-m", info.model_uri,
     "--host", "127.0.0.1", "--port", "5002", "--env-manager", "local"],
    env={**os.environ, "MLFLOW_TRACKING_URI": "http://127.0.0.1:5001"},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

up = False
for _ in range(90):                       # the model loads on startup; give it time
    try:
        if requests.get(f"{SERVE_URL}/health", timeout=2).status_code == 200:
            up = True; break
    except requests.exceptions.RequestException:
        pass
    time.sleep(1)
print("serving healthy:", up)

## Step 5 — call `/invocations`

POST a JSON body with an `inputs` list; you get `{"predictions": [...]}` back. From the shell:

```bash
curl -X POST http://127.0.0.1:5002/invocations \
  -H 'Content-Type: application/json' \
  -d '{"inputs": ["What is MLflow Tracing?"]}'
```

Or from Python (run this cell once the server above is up):

In [4]:
import requests

resp = requests.post(
    "http://127.0.0.1:5002/invocations",
    json={"inputs": ["What does the MLflow model registry do?",
                     "What is MLflow Tracing?"]},
    timeout=120,
)
resp.raise_for_status()
print(resp.json())

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=5002): Max retries exceeded with url: /invocations (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=5002): Failed to establish a new connection: [Errno 111] Connection refused"))

In [ ]:
# Stop the background server when you're done (Option A only).
server.terminate()
server.wait(timeout=10)
print("server stopped")

> **If the kernel crashes before `teardown` runs**, the server subprocess keeps holding port 5002. Check for it with `lsof -i:5002` (or `ss -ltnp | grep 5002`), then either re-run the teardown cell or kill it directly:

```bash
kill $(lsof -t -i:5002)
```

## Promotion reaches the endpoint — no redeploy

Because the served app calls `load_prompt("prompts:/qa-answer@production")` **on every request**, you can improve the prompt without touching the server:

1. Register a better prompt version (as in `e_prompt_registry`).
2. Move the alias: `mlflow.genai.set_prompt_alias("qa-answer", "production", <new_version>)`.
3. The next `/invocations` call uses the new prompt — the running server picks it up.

This is the GenAI version of `ml/g_model_serving`'s *"promotion reaches the endpoint"*: there, moving a model `@champion` alias changed what got served; here, moving a prompt `@production` alias does. (If you'd rather pin the prompt at deploy time, load it in `log_context`/at log time instead — the trade-off is reproducibility vs. live updates.)

## Next steps

- **Serve the `d_` agent the same way.** A LangChain agent logs via `mlflow.langchain.log_model(lc_model="agent.py", ...)` (models-from-code) and serves with the identical `mlflow models serve` command — swap the model script, keep the workflow.
- **`g_feedback_and_monitoring.ipynb`** (planned, stretch) — collect human feedback / assessments on the traces this endpoint produces, and run scorers on live traffic.
- **Containerize:** `mlflow models build-docker` packages this endpoint into an image, exactly as in `ml/g_model_serving`.